In [66]:
def principales(df):
    '''
    Devuelve la lista de las enzimas por las que se metaboliza y las principales en todo caso de haber

    Parametros
    ------------------
        df- Solo incluye las filas con el ppio activo concreto que estemos consultando
    Devolución
    -----------------------------
        enz - Lista de enzimas por las que se metaboliza el principio activo
        ppal - Lista de enzimas principales por las que se metaboliza el fármaco (prioridad=1)
    '''
    #df con todas las enzimas (no target por las que se metaboliza)
    df_enzimas = df[df["Tipo"]=="enzyme" ]
    # Lista de enzimas por las que se metaboliza el ppio
    enz = df_enzimas["Gene_name"].unique().tolist()      
    #Lista de enzimas principales (si tiene) el ppip
    ppal = df_enzimas[df_enzimas["Prioridad"]==1]["Gene_name"].unique().tolist()

    return enz, ppal


# ALGUN Q OTRO INTENTO + TARDE

In [ ]:
#LODELSTRIP()LOWER()
#Igual en vez de true/false para los booleanos deberia devolver los codigos atc y los ppios o algo asi VER A LA VUELTA
#RECUPERAR EL CALCULAR RIESGO Y PONERLO TODO EN FUNCIONES.PY O ALGO ASI
def interaccion (ppio_1, ppio_2, DDI, efectos_adversos, texto=False):
    '''
    Funcion que determina si existe interacción entre los dos principios activos o no

    Parámetros
    -----------------
        ppio_1 - El primer principio activo que se quiere comparar con el segundo
        ppio_2 - El segundo principio activo que se quiere comparar con el primero
        DDI - Dataframe que contiene los nombres, ATC, enzimas, target, acciones y prioridad
        efectos_adversos - Dataframe que contiene los efectos adversos de cada ppio, el ATC y la frecuencia con la que aparecen
        texto - Si es necesaria la impresión de texto o no (para comparar el ATC no lo es), por defecto a False
    Devolucion
    ------------------
        Booleano (True cuando hay interacción False cuando no existe(aunque se describe como leve)#DECIRLO AL FINAL)
    '''
    
    #Comprobamos primero que este en nuestra BBDD
    if (ppio_1 in DDI["Drug_name"].values) and (ppio_2 in DDI["Drug_name"].values):
        df_1 = DDI[DDI["Drug_name"] == ppio_1]
        enz_1, ppal_1 = principales(df_1)
        
        df_2 = DDI[DDI["Drug_name"] == ppio_2]
        enz_2, ppal_2 = principales(df_2)
    else: 
        #CAMBIA ESTO PPIO NO ES NADA
        print(f"{ppio} no se ha encontrado en la base de datos, porfavor introduce otro")
        break
            
    #Sabiendo que está, si queremos texto o no (descripcion de las enz y los ppios)
    if texto:
        texto_intro(ppio_1, enz_1, ppal_1, True)
        texto_intro(ppio_2, enz_2, ppal_2)

    #Aqui compruebo si hay interaccion o no
    coincidentes = list(set(ppal_1) & set(ppal_2))
    #Si hay interaccion entre ellas se mostrará en coincidentes, sino, la lista será vacia
    if coincidentes:
        #Sacar las acciones y el texto sera para cada enzima por separado
        if texto:
            for e in coincidentes:
                #La fila q contiene la enzima que se quiere ver
                fila_1 = df_1[df_1["Gene_name"]==e]
                #Separamos por si tiene | (no da error si no lo tiene)
                separado_1 = fila_1["Accion"].str.split(r"\|")
                #Nos quedamos con los distintos
                acciones_1 = set(separado_1.explode().tolist())
    
                #Lo mismo pero con el otro principio consultado
                fila_2 = df_2[df_2["Gene_name"]==e]
                separado_2 = fila_2["Accion"].str.split(r"\|")
                acciones_2 = set(separado_2.explode().tolist())

                #Funcion que contiene el texto
                texto_acciones(ppio_1,acciones_1,ppio_2,acciones_2,e) #PARA CADA ENZIMA
                
        return True

    #Lista vacía: sin interacción
    return False

In [69]:
for i in ["SI"]:
    print(i)

SI



- Que hago cuando hay mas de un codigo ATC con el que comparar q tiene los 5 primeros diferentes
    COMPARAS CON LOS 5 PRIMEROS, SI LOS HAY DIFERENTES DOS LISTAS Y SI NO HAY COINCIDENCIAS CON LOS 4 PRIMEROS INDICANDO PRIMERO DE QUE CODIGO ATC SALE
- Lo de los efectos adversos
  SOLO IMPRIMIR
- Deberia subir al git los notebooks de prueba?
  NO
- Cuando vuelvo a buscar opciones tengo que comparar cada una de las opciones con cada uno de los farmacos? Es decir: si solo quito uno? O todas las posibles opciones con todas las posibles opciones? que me puede llegar a parecer mas facil, pero entonces como diferencio que farmaco viene de que?
  SIN SOLUCION MUCHOS BUCLES Y MUCHOS IFELSE


Reducido a solo combinar dos ppio_act

In [ ]:
#Usarlo para combinatoria de ATC
from itertools import product

lista1 = [1, 2]
lista2 = ['X', 'Y']

for comb in product(lista1, lista2):
    print(comb)

# Salida: (1, 'X') | (1, 'Y') | (2, 'X') | (2, 'Y')


In [3]:
import pandas as pd
DDI = pd.read_csv("DDI_sea.csv")
efectos = pd.read_csv("efectos_adversos.csv")

In [4]:
efectos.head()

,Drug_ATC,Drug_name,Side_effect,Freq_media
0,A16AA01,carnitine,Abdominal pain,0.116000
1,A16AA01,carnitine,Gastrointestinal pain,0.116000
2,A16AA01,carnitine,Amblyopia,0.036667
3,A16AA01,carnitine,Anaemia,0.058000
4,A16AA01,carnitine,Decreased appetite,0.042500


# CUANDO HAY UNA LISTA DE ACCIONES MAYOR A UNO Y PON Q HAYA A LA VEZ INHIBIDOR E INDUCTOR PRIMERO COMPARO Q SI LO HAY EN LOS DOS Y ASI DIGO MISMA INTERACCION, DONE

In [39]:
#ES UNA LISTA DE LISTAS, VER ESTO
problema = DDI[DDI["Drug_name"]=="Omeprazole"]
fila = problema[problema["Gene_name"]=="CYP3A4"]
acc = fila["Accion"].str.split(r"\|")
acciones = set(acc.explode().tolist())

posibles = ["inducer", "substrate", "inhibitor"]



resultado = [x for x in acciones if x in posibles]

print(resultado)  

['inducer', 'substrate', 'inhibitor']


In [41]:
#Cuando son None NO  HAY NONE :)
nones = DDI[DDI["Tipo"]=="Enzyme"]
nones.isna().any()

DrugBank_id    False
Drug_ATC       False
Drug_name      False
Tipo           False
Gene_name      False
Accion         False
Prioridad      False
dtype: bool

In [ ]:
DDI.head(40)

# COMO HACER LA SEPARACION CON ENZIMAS
Poner def supongo

In [63]:
if ("Cetuximab" in DDI["Drug_name"].values):
    print("si")
else:
    print("NO")

si


In [181]:
uno = DDI[DDI["Drug_name"] == "Clopidogrel"]
dos = DDI[DDI["Drug_name"] == "Omeprazole"]
uno
dos
#Hacer comprobacion primero de si no es nulo
p_uno = uno[uno["Tipo"]=="enzyme" ]["Gene_name"].tolist()
p_dos = dos[dos["Tipo"]=="enzyme" ]["Gene_name"].tolist()
lista= list(set(p_uno) & set(p_dos))

In [217]:
uno

,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad
8657,DB00758,B01AC04,Clopidogrel,target,P2RY12,antagonist,0.0
8658,DB00758,B01AC04,Clopidogrel,enzyme,CYP2C9,substrate|inhibitor,0.0
8659,DB00758,B01AC04,Clopidogrel,enzyme,CYP1A2,substrate,0.0
8660,DB00758,B01AC04,Clopidogrel,enzyme,CYP3A4,substrate,0.0
8661,DB00758,B01AC04,Clopidogrel,enzyme,CYP3A5,substrate,0.0
8662,DB00758,B01AC04,Clopidogrel,enzyme,CYP2C19,substrate,1.0


In [242]:
dos

,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad
3612,DB00338,A02BC51,Omeprazole,target,ATP4A,inhibitor,0.0
3613,DB00338,A02BC01,Omeprazole,target,ATP4A,inhibitor,0.0
3614,DB00338,A02BD05,Omeprazole,target,ATP4A,inhibitor,0.0
3615,DB00338,A02BD16,Omeprazole,target,ATP4A,inhibitor,0.0
3616,DB00338,A02BD01,Omeprazole,target,ATP4A,inhibitor,0.0
3617,DB00338,A02BC51,Omeprazole,target,ATP4B,modulator,0.0
3618,DB00338,A02BC01,Omeprazole,target,ATP4B,modulator,0.0
3619,DB00338,A02BD05,Omeprazole,target,ATP4B,modulator,0.0
3620,DB00338,A02BD16,Omeprazole,target,ATP4B,modulator,0.0
3621,DB00338,A02BD01,Omeprazole,target,ATP4B,modulator,0.0


Para buscar su es sustrato, inhibidor o modulador buscar lo de in cadena de caracteres porque hay algunos que tienen | y tener en cuenta que hay otros que tienen el elemento de accion en nulo.  
Tener en cuenta que lo de sustrato/inhibidor/lo que sea es solo para el texto de explicacion, hacer funciones xra el asunto :)

In [ ]:
#Lo de los efectos adversos iria en esta funcion en mi humilde opinion, igual solo en el de riesgo bajo porque termina el algoritmo técnicamente
def texto_niveles(acc_1, acc_2, ppio_1, ppio_2, nivel, aux=None):
    '''
    Texto que describe la interaccion y accion que tiene cada principio activo inhibitor/substrate/inducer

    Parámetros
    ---------------------
        acc_1 - Acciones del ppio_1 dispuestas en una lista (solo inhibitor/substrate/inducer/None)
        acc_2 - Acciones del ppio_2 dispuestas en una lista (solo inhibitor/substrate/inducer/None)
        ppio_1 - Nombre del primer principio activo
        ppio_2 - Nombre del segundo principio activo
        nivel - Variable que puede tomar 3 valores: "Alto", "Medio" y "Bajo"
        aux - Valor que indica (en caso de ser nivel medio) cual de los dos tiene prioridad, en caso de no ser ninguno None

    Devolución
    -------------------------
        None
    '''
    print("Las siguientes consideraciones se toman teniendo en cuenta que se trata de la primera vez que la persona consume ambos principios activos debido a que se desconocen con exactitud los efectos a largo plazo que puede tener el paciente dependiendo del tiempo de ingesta")
    #Alguna de las dos listas esta vacía y por tanto no tenemos en cuenta
    if (len(acc_1)==0 or len(acc_2)==0):
        print("Alguna de las acciones de estos dos principios activos consultados no constan en la base de datos consultada o no actuan como inhibidor, inductor o sustrato. Con lo que no se ha podido determinar ninguna prevalencia")
    
    #Los dos son 1
    elif nivel == "Alto":
        #Ver si da error cuando la lista solo es un valor, en ambos casos
        if (x in acc_2 for x in acc_1):
            print("Como ambos principios activos actúan de la misma manera competirán al mismo nivel y por tanto ninguno tiene prioridad de metabolización frente al otro") #Es decir, como {poner aqui sustrato/inhibidor/inductor, me lo daria la lista de arriba}
        else:
            #Sustrato-inhibidor
            if ("substrate" in acc_1) and ("inhibitor" in acc_2):
                print(f"Para este tipo de interacción en el que uno de los principios activos es sustrato y otro inhibidor prevalecería el inhibidor, que en este caso concreto corresponde al principio activo: {ppio_2}")
            elif ("substrate" in acc_2) and ("inhibitor" in acc_1):
                print(f"Para este tipo de interacción en el que uno de los principios activos es sustrato y otro inhibidor prevalecería el inhibidor, que en este caso concreto corresponde al principio activo: {ppio_1}")
            #Sustrato-inductor
            elif ("substrate" in acc_1) and ("inducer" in acc_2):
                print(f"Para este tipo de interacción en el que uno de los principios activos es sustrato y otro inductor prevalecería el sustrato, que en este caso concreto corresponde al principio activo: {ppio_1}")
            elif ("substrate" in acc_2) and ("inducer" in acc_1):
                print(f"Para este tipo de interacción en el que uno de los principios activos es sustrato y otro inductor prevalecería el sustrato, que en este caso concreto corresponde al principio activo: {ppio_2}")
            #Inhibidor-inductor
            elif ("inhibitor" in acc_1) and ("inducer" in acc_2):
                print(f"Para este tipo de interacción en el que uno de los principios activos es innhibidor y otro inductor prevalecería el inhibidor, que en este caso concreto corresponde al principio activo: {ppio_1}")
            elif ("inhibitor" in acc_2) and ("inducer" in acc_1):
                print(f"Para este tipo de interacción en el que uno de los principios activos es innhibidor y otro inductor prevalecería el inhibidor, que en este caso concreto corresponde al principio activo: {ppio_2}")

    elif nivel == "Medio":
        #Uno es prioritario, el otro no sabemos
        if aux is not None:
            if aux == ppio_1:
                
            else:
        #No sabemos prioridad de ninguno de los dos fármacos
        else:

    elif nivel == "Bajo":
        print("Ninguno de los dos principios activos se metabolizan por la misma enzima dentro de las 6 contempladas: CYP3A4, CYP2C9, CYP2D6, CYP1A2, CYP2C19, CYP3A5")

In [135]:
def comparacion (df_1,df_2):
    enzima_1 = df_1[df_1["Tipo"]=="enzyme" ]["Gene_name"].tolist()
    enzima_2 = df_2[df_2["Tipo"]=="enzyme" ]["Gene_name"].tolist()
    coincidentes = list(set(enzima_1) & set(enzima_2))
    return coincidentes


#ESTA ME COMPARA TODAS LAS ENZIMAS NO LAS PRINCIPALES

In [203]:
#Dentro de este van los textos de los niveles
def calcular_riesgo (coincidentes, ppio_act, comparo, principal, segundo):
    #RIESGOS ALTOS
    #Como puede haber mas de uno principal
    if len(principal)>1 and len(segundo)>1 and list(set(principal) & set(segundo)):
        
        print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    #Principal es lista
    elif len(principal)>1 and segundo[0] in principal:
        print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    elif len(segundo)>1 and principal[0] in segundo::
        print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    #Si es la de los dos ALTO (dos principales) estara en coincidentes si llega hasta aqui
    elif principal[0] == segundo[0] :
        print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")          #Describirlo

    
    #RIESGO MEDIO
    #Si es solo la de uno o no hay prioridad en ningun ppio_act MEDIO
    else:
        print(f"Riesgo de interacción entre {ppio_act} y {comparo} MEDIO")       #Describirlo
        if ():
        elif():
        else:

In [240]:
def busco_opciones(DDI,ppio_ini, ATC_ini, ppio_comp, ATC_comp):
    opciones = {}
    ATC = [ATC_ini, ATC_comp]
    for i in range(len(ATC)):
        codigo_referencia = ATC[i][:5]
        resultado = DDI[DDI['Drug_ATC'].str.startswith(codigo_referencia, na=False)]["Drug_name"].unique().tolist()
        if i == 0:
            opciones[ppio_ini] = resultado
        else:
            opciones[ppio_comp] = resultado

In [205]:
prescripcion = ["Omeprazole","Clopidogrel"]

In [207]:
for i in range(len(prescripcion)):
    for j in range(i + 1, len(prescripcion)):

        ppio_act = prescripcion[i]
        comparo = prescripcion[j]
        
        #Me va a dar error si no es talcual asiq split().lower() deberia
        inicial = DDI[DDI["Drug_name"] == ppio_act]
        secundario = DDI[DDI["Drug_name"] == comparo]
        
        #Comparo ambos
        coincidentes = comparacion(inicial, secundario)

        #Vemos las vias principales
        principal = inicial[inicial["Prioridad"]==1.0]["Gene_name"].tolist()
        segundo = secundario[secundario["Prioridad"]==1.0]["Gene_name"].tolist()
        #Cuando calculamos el riesgo no devolvemos nada, solo texto
        if coincidentes:
            calcular_riesgo(coincidentes, ppio_act, comparo, principal, segundo)
            #En esta linea busco las opciones
            opciones = busco_opciones(DDI,ppio_ini, ATC_ini, ppio_comp, ATC_comp)

        
        else:
            #Cuando no coinciden es interaccion BAJA o leve
            print(f"Riesgo de interacción entre {ppio_act} y {comparo} BAJO")

llego
Riesgo de interaccion entre Omeprazole y Clopidogrel ALTO
